# 📊 Notebook 01 — Dataset Generation & Data Cleaning
**FR1 + FR2 | AI Subscription & Auto-Debit Intelligence System**
**Team 7 — Mansi & Samyak**

This notebook covers:
- FR1: Synthetic dataset generation using Faker (50K–200K transactions)
- Null/missing value distribution before cleaning
- FR2: Preprocessing steps with before/after comparison
- Visual analytics on the cleaned dataset


In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11
sns.set_theme(style='whitegrid')
print('Libraries loaded ✅')

## 1. FR1 — Synthetic Dataset Generation

In [ ]:
from modules.fr1_dataset_generator import generate_dataset

# Generate 50K–200K transactions (1200 accounts × ~100 txns each)
df_raw = generate_dataset(n_accounts=1200)
print(f'Dataset shape  : {df_raw.shape}')
print(f'Columns        : {list(df_raw.columns)}')
df_raw.head(5)

In [ ]:
# ── Distribution of transaction types ──
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1: Subscription vs Non-subscription
sub_counts = df_raw['SubscriptionFlag'].value_counts()
axes[0].bar(['Non-Subscription','Subscription'], sub_counts.values,
            color=['#3498db','#e74c3c'], edgecolor='white')
axes[0].set_title('FR1: Subscription vs Non-Subscription', fontweight='bold')
axes[0].set_ylabel('Count')
for i,(v,c) in enumerate(zip(sub_counts.values, ['#3498db','#e74c3c'])):
    axes[0].text(i, v+500, f'{v:,}', ha='center', fontweight='bold')

# 2: Transaction status (Success vs Failed)
status = df_raw['Status'].value_counts()
axes[1].bar(status.index, status.values, color=['#2ecc71','#e74c3c'], edgecolor='white')
axes[1].set_title('Transaction Status (Failed Debits)', fontweight='bold')
axes[1].set_ylabel('Count')
for i,(idx,v) in enumerate(status.items()):
    axes[1].text(i, v+300, f'{v:,}', ha='center', fontweight='bold')

# 3: Frequency breakdown
freq = df_raw[df_raw['SubscriptionFlag']==1]['Frequency'].value_counts()
axes[2].bar(freq.index, freq.values, color=['#9b59b6','#16a085','#f39c12'], edgecolor='white')
axes[2].set_title('Subscription Frequency Distribution', fontweight='bold')
axes[2].set_ylabel('Count')

plt.suptitle('FR1: Synthetic Dataset Overview', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../reports/nb01_dataset_overview.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Failed debits : {(df_raw["Status"]=="FAILED").sum():,} ({(df_raw["Status"]=="FAILED").mean()*100:.1f}%)')

In [ ]:
# ── Amount distribution by merchant ──
top_merchants = df_raw[df_raw['SubscriptionFlag']==1].groupby('Merchant')['Amount'].mean().sort_values(ascending=False).head(10)
fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.barh(top_merchants.index[::-1], top_merchants.values[::-1], color='#2980b9', edgecolor='white')
ax.set_xlabel('Average Amount (₹)')
ax.set_title('Top 10 Merchants by Average Subscription Amount', fontweight='bold')
for bar, val in zip(bars, top_merchants.values[::-1]):
    ax.text(bar.get_width()+20, bar.get_y()+bar.get_height()/2, f'₹{val:,.0f}', va='center', fontweight='bold')
plt.tight_layout(); plt.show()

## 2. FR2 — Data Cleaning: Null Analysis & Preprocessing

In [ ]:
# ── NULL ANALYSIS BEFORE CLEANING ──
null_before = df_raw.isnull().sum()
null_pct    = (null_before / len(df_raw) * 100).round(2)
null_report = pd.DataFrame({'Null Count': null_before, 'Null %': null_pct})
null_report = null_report[null_report['Null Count'] > 0]

print('=== NULL VALUES BEFORE CLEANING ===')
print(null_report.to_string())
print(f'\nTotal null cells : {null_before.sum():,}')

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(null_report.index, null_report['Null %'], color=['#e74c3c','#f39c12','#3498db'])
ax.set_title('Null Value % Before Cleaning (FR2)', fontweight='bold')
ax.set_ylabel('Null Percentage (%)')
for i, (col, pct) in enumerate(zip(null_report.index, null_report['Null %'])):
    ax.text(i, pct+0.02, f'{pct:.2f}%', ha='center', fontweight='bold')
ax.axhline(y=0, color='black', linewidth=0.5)
plt.tight_layout(); plt.show()

In [ ]:
from modules.fr2_data_cleaning import clean_data

df_clean = clean_data(df_raw)

# ── BEFORE vs AFTER comparison ──
null_after = df_clean.isnull().sum()
comparison = pd.DataFrame({
    'Before': null_before[['Description','Amount','Balance']],
    'After' : null_after[['Description','Amount','Balance']]
})
print('=== BEFORE vs AFTER CLEANING ===')
print(comparison)

x    = range(len(comparison))
width= 0.35
fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar([i-width/2 for i in x], comparison['Before'], width, label='Before', color='#e74c3c', edgecolor='white')
b2 = ax.bar([i+width/2 for i in x], comparison['After'],  width, label='After',  color='#2ecc71', edgecolor='white')
ax.set_xticks(list(x)); ax.set_xticklabels(comparison.index)
ax.set_title('Null Values: Before vs After Cleaning (FR2)', fontweight='bold')
ax.set_ylabel('Null Count'); ax.legend()
for bar in b1:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+10, str(int(bar.get_height())), ha='center', fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# ── Description normalisation samples ──
print('=== DESCRIPTION NORMALISATION (FR2) ===')
print(f'{"Original":<45} {"Cleaned"}')
print('-'*90)
samples = df_clean[['Description','Description_Clean']].drop_duplicates().head(15)
for _, row in samples.iterrows():
    if row['Description'] != row['Description_Clean']:
        print(f'{str(row["Description"]):<45} {row["Description_Clean"]}')

In [ ]:
# ── Monthly transaction volume over time ──
df_clean['Date'] = pd.to_datetime(df_clean['Date'])
monthly = df_clean.groupby(df_clean['Date'].dt.to_period('M')).size().reset_index()
monthly.columns = ['Month','Count']
monthly['Month'] = monthly['Month'].astype(str)

fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(range(len(monthly)), monthly['Count'], alpha=0.3, color='#2980b9')
ax.plot(range(len(monthly)), monthly['Count'], color='#2980b9', lw=2.5, marker='o', markersize=4)
ax.set_xticks(range(0, len(monthly), 2))
ax.set_xticklabels(monthly['Month'].iloc[::2], rotation=45)
ax.set_title('Monthly Transaction Volume — Cleaned Dataset (FR2)', fontweight='bold')
ax.set_ylabel('Transaction Count'); ax.grid(axis='y', alpha=0.4)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.show()

print(f'\nDataset Summary after FR2 Cleaning:')
print(f'  Total rows    : {len(df_clean):,}')
print(f'  Subscriptions : {df_clean["SubscriptionFlag"].sum():,} ({df_clean["SubscriptionFlag"].mean()*100:.1f}%)')
print(f'  Date range    : {df_clean["Date"].min().date()} → {df_clean["Date"].max().date()}')
print(f'  Null values   : {df_clean[["Description","Amount","Balance"]].isnull().sum().sum()} ✅')

## ✅ Summary

| Step | Result |
|------|--------|
| FR1 Dataset | 134,000+ transactions generated with Faker |
| Null in Amount | Filled with merchant-median imputation |
| Null in Balance | Forward-filled within each CustomerID |
| Description | Normalised + expanded abbreviations |
| Duplicates | Removed by TransactionID |
